# 02 — Model Architecture
Inspect the U-Net, count parameters, verify forward pass shapes.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
from src.model.unet import ConditionalUNet
from src.model.diffusion import SprayChartDiffusion

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
NUM_BATTERS = 1000   # placeholder — use actual count from batter_id_map.json

unet = ConditionalUNet(
    num_batters=NUM_BATTERS,
    base_channels=64,
    channel_multipliers=(1, 2, 4, 8),
    inpaint_mode=False,
).to(device)

total_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'U-Net parameters: {total_params/1e6:.2f}M')

In [ ]:
# Verify forward pass
B = 4
x_t = torch.randn(B, 1, 64, 64, device=device)
t   = torch.randint(0, 1000, (B,), device=device)
bat = torch.randint(0, NUM_BATTERS, (B,), device=device)
sit = torch.randint(0, 13, (B,), device=device)   # 0-12

with torch.no_grad():
    eps_hat = unet(x_t, t, bat, sit)

print('Output shape:', eps_hat.shape)   # expected: (4, 1, 64, 64)

In [ ]:
# Inpainting mode: input channel = 2
unet_inp = ConditionalUNet(num_batters=NUM_BATTERS, inpaint_mode=True).to(device)
partial  = torch.randn(B, 1, 64, 64, device=device)

with torch.no_grad():
    eps_hat_inp = unet_inp(x_t, t, bat, sit, partial_image=partial)

print('Inpaint output shape:', eps_hat_inp.shape)   # expected: (4, 1, 64, 64)

In [ ]:
# Full diffusion model — loss forward pass
model = SprayChartDiffusion(unet=unet).to(device)
x_0 = torch.rand(B, 1, 64, 64, device=device)   # fake density images

loss = model.loss(x_0, bat, sit)
print(f'DDPM loss: {loss.item():.4f}')

In [ ]:
# Print model structure summary
for name, module in unet.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f'{name:30s}  {params/1e6:.2f}M params')